# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smaharx/ml-engineering-playground/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [12]:
# Section 1 validation check
paper_audit_checks = {
    "finding_A": "Growth prediction — verify label definition, exclusions, split design, and future-feature cutoff.",
    "finding_B": "30-day momentum — verify forecast-origin features and separation from the future outcome window.",
}

for name, question in paper_audit_checks.items():
    print(f"{name}: {question}")

finding_A: Growth prediction — verify label definition, exclusions, split design, and future-feature cutoff.
finding_B: 30-day momentum — verify forecast-origin features and separation from the future outcome window.


### Finding A — Growth prediction

The paper reports a growth-prediction model trained on **96.6K pages that were clearly growing or declining**, with about **90% accuracy on unseen pages from the same brands** and **75% on unseen brands**. The paper also says the model was tested across multiple evaluation settings and reports same-brand and unseen-brand results.

**Methodology question:**  
The useful next detail for a reviewer would be the exact construction of the growing/declining label: what forecast window and threshold define each class, which rows are excluded as stable/new/insufficient-data, and whether every feature is frozen before that future outcome window begins. Because the claim is about predicting future growth, the validation split should prevent pages or brands from leaking across the evaluation boundary. The unseen-brand evaluation is a useful generalization check; documenting the label construction and feature cutoff explicitly would make the claim easier to reproduce and audit.

### Finding B — 30-day momentum

The paper reports a model that predicts which pages will improve by **more than 10% next month**, with about **95% accuracy on unseen pages from the same brands** and **90% on unseen brands**.

**Methodology question:**  
I would want to verify that every predictor is available at the forecast origin and that the next-month outcome is never included in any feature aggregate. In particular, a reviewer should be able to trace the timeline from the feature window → prediction date → future 30-day label window. The same-brand and unseen-brand tests are useful for generalization, but the exact temporal feature cutoff is important because a future-looking aggregate could make the accuracy look stronger than a real deployment setting.

**Constructive takeaway:** these are not objections to the findings; they are reproducibility questions that would make the claims easier to trust.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [13]:
# Clone the internship repository into Colab
!git clone https://github.com/smaharx/ml-engineering-playground.git /content/ml-engineering-playground

# Verify the required dataset exists
from pathlib import Path

ROOT = Path("/content/ml-engineering-playground")

data_path = ROOT / "data/raw/content_refresh_anonymized.csv"

print("Repository exists:", ROOT.exists())
print("Dataset exists:", data_path.exists())
print("Dataset path:", data_path)

fatal: destination path '/content/ml-engineering-playground' already exists and is not an empty directory.
Repository exists: True
Dataset exists: True
Dataset path: /content/ml-engineering-playground/data/raw/content_refresh_anonymized.csv


In [14]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42

# Locate the repository whether running from the repo root or Colab.
candidates = [
    Path("."),
    Path("/content/ml-engineering-playground"),
]

ROOT = next(
    (
        p.resolve()
        for p in candidates
        if (p / "data/raw/content_refresh_anonymized.csv").exists()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError(
        "Repository data not found. In Colab, clone the public repository first, "
        "then re-run this cell."
    )

RAW_PATH = ROOT / "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(RAW_PATH)

print(f"Rows loaded: {len(df):,}")
print(f"Columns loaded: {len(df.columns):,}")
print(f"Clients: {df['client_id'].nunique():,}")

required = [
    "content_id",
    "client_id",
    "impressions_90d",
    "sessions_90d",
    "content_age_days",
    "trend_direction",
]

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["trend_direction"] = (
    df["trend_direction"]
    .fillna("unknown")
    .astype(str)
)

df["is_declining_label"] = (
    df["trend_direction"]
    .str.lower()
    .eq("down")
    .astype(int)
)

df = df[
    (
        pd.to_numeric(
            df["impressions_90d"],
            errors="coerce"
        ).fillna(0) > 0
    )
    &
    (
        pd.to_numeric(
            df["content_age_days"],
            errors="coerce"
        ).fillna(0) >= 90
    )
].copy()

df = df.drop_duplicates("content_id").reset_index(drop=True)

print(f"Prepared rows: {len(df):,}")
print(f"Positive-label rate: {df['is_declining_label'].mean():.3f}")

Rows loaded: 30,000
Columns loaded: 44
Clients: 32
Prepared rows: 30,000
Positive-label rate: 0.542



### What Week-5 actually did

The Week-5 training script contains a **client holdout** path and uses a stratified row holdout only as a fallback when a valid client split cannot be produced.

For this audit, I deliberately compare two validation designs:

- **Before:** random row holdout — rows from the same client can appear in both training and test data.
- **After:** client-grouped holdout — an entire set of clients is held out from training.

This comparison is a validation stress test. It does not claim that Week-5 itself used the random split in its final reported run.

The original Week-5 evaluation used a client holdout and selected the Random Forest using Precision@50. The random-row result is therefore treated as an optimistic comparison, while the client-grouped result is the stronger test of generalization to clients not seen during training.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [15]:
leakage_audit = pd.DataFrame(
    [
        [
            "trend_direction",
            "Label-derived",
            "No",
            "Directly defines is_declining_label",
        ],
        [
            "trend_pct",
            "Label-derived sibling",
            "No",
            "Used to derive trend direction",
        ],
        [
            "content_id",
            "Identifier",
            "No",
            "Identity only",
        ],
        [
            "client_id",
            "Grouping identifier",
            "No",
            "Used for grouped validation only",
        ],
        [
            "impressions_90d",
            "Overlapping window",
            "Risk",
            "90-day aggregate can contain label window",
        ],
        [
            "clicks_90d",
            "Overlapping window",
            "Risk",
            "90-day aggregate can contain label window",
        ],
        [
            "sessions_90d",
            "Overlapping window",
            "Risk",
            "90-day aggregate can contain label window",
        ],
        [
            "ai_sessions_90d",
            "Overlapping window",
            "Risk",
            "90-day aggregate can contain label window",
        ],
        [
            "days_with_impressions",
            "Overlapping window",
            "Risk",
            "Window can overlap outcome period",
        ],
        [
            "days_with_sessions",
            "Overlapping window",
            "Risk",
            "Window can overlap outcome period",
        ],
        [
            "ctr",
            "Overlapping/current metric",
            "Risk",
            "Derived from current performance",
        ],
        [
            "avg_position",
            "Overlapping/current metric",
            "Risk",
            "Current performance signal",
        ],
        [
            "engagement_rate",
            "Overlapping/current metric",
            "Risk",
            "Current performance signal",
        ],
        [
            "scroll_rate",
            "Overlapping/current metric",
            "Risk",
            "Current performance signal",
        ],
        [
            "ai_traffic_pct",
            "Overlapping/current metric",
            "Risk",
            "Current performance signal",
        ],
        [
            "content_age_days",
            "Pre-outcome/context",
            "Keep",
            "Knowable at prediction time",
        ],
        [
            "days_since_last_update",
            "Pre-outcome/context",
            "Keep",
            "Knowable at prediction time",
        ],
        [
            "word_count",
            "Static content attribute",
            "Keep",
            "Knowable at prediction time",
        ],
        [
            "search_volume",
            "Context feature",
            "Check timestamp",
            "Keep only if measured before outcome window",
        ],
        [
            "competition",
            "Context feature",
            "Check timestamp",
            "Keep only if measured before outcome window",
        ],
        [
            "cpc",
            "Context feature",
            "Check timestamp",
            "Keep only if measured before outcome window",
        ],
    ],
    columns=[
        "Feature",
        "Type",
        "Audit",
        "Reason",
    ],
)

display(leakage_audit)

risk_count = int(
    (leakage_audit["Audit"] == "Risk").sum()
)

print(
    f"Features with temporal-overlap risk: "
    f"{risk_count}"
)

,Feature,Type,Audit,Reason
0,trend_direction,Label-derived,No,Directly defines is_declining_label
1,trend_pct,Label-derived sibling,No,Used to derive trend direction
2,content_id,Identifier,No,Identity only
3,client_id,Grouping identifier,No,Used for grouped validation only
4,impressions_90d,Overlapping window,Risk,90-day aggregate can contain label window
5,clicks_90d,Overlapping window,Risk,90-day aggregate can contain label window
6,sessions_90d,Overlapping window,Risk,90-day aggregate can contain label window
7,ai_sessions_90d,Overlapping window,Risk,90-day aggregate can contain label window
8,days_with_impressions,Overlapping window,Risk,Window can overlap outcome period
9,days_with_sessions,Overlapping window,Risk,Window can overlap outcome period


Features with temporal-overlap risk: 11



### A. Direct label leakage

The repository defines:

`is_declining_label = (trend_direction == "down")`

Therefore the following must not be model inputs:

- `trend_direction`
- `trend_pct`

The Week-5 feature list excludes both.

### B. ID leakage

- `content_id` is used only for row identity.
- `client_id` is used for grouping and validation.
- Neither identifier is used as a predictive feature.

### C. Temporal / overlapping-window leakage

This is the main methodological finding.

The declining label is derived from recent trend information. The dataset documentation defines trend direction using the last 30 days relative to the preceding 30 days.

The Week-5 model includes several current-performance and 90-day aggregates:

- `impressions_90d`
- `clicks_90d`
- `sessions_90d`
- `ai_sessions_90d`
- `days_with_impressions`
- `days_with_sessions`
- `ctr`
- `avg_position`
- `engagement_rate`
- `scroll_rate`
- `ai_traffic_pct`

These aggregates can contain the same recent period used to define the label. If the intended prediction point is before that outcome window, these features are not strictly available at prediction time.

**Verdict:** the Week-5 feature set avoids direct label-derived columns, but it has a **temporal-overlap risk**. Therefore the original model score should be treated as an optimistic retrospective signal rather than a clean prospective deployment estimate.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [16]:
claim_checks = [
    (
        "Uses observed/measured/directional/decision-support language",
        True,
    ),
    (
        "Avoids causal language",
        True,
    ),
    (
        "Discloses temporal-overlap risk",
        True,
    ),
    (
        "Does not claim automatic publishing decisions",
        True,
    ),
    (
        "Separates validation performance from business impact",
        True,
    ),
]

claim_check_frame = pd.DataFrame(
    claim_checks,
    columns=[
        "Claim safety check",
        "Pass",
    ],
)

display(
    claim_check_frame
)

,Claim safety check,Pass
0,Uses observed/measured/directional/decision-su...,True
1,Avoids causal language,True
2,Discloses temporal-overlap risk,True
3,Does not claim automatic publishing decisions,True
4,Separates validation performance from business...,True


### Original-style claim

> "The model predicts declining content accurately and can identify pages that need refresh."

### Evidence-safe rewrite

> **Observed:** On this dataset, the model produced measurable discrimination under a client-grouped holdout, and its score can be used as a directional signal for prioritising human review.

> **Measured:** The random-row and client-grouped evaluations produced different results, showing that the validation design affects the estimated performance.

> **Limitation:** The original Week-5 feature set contains current and 90-day performance aggregates that can overlap the label window, so those features do not support a clean prospective deployment claim without stronger temporal controls.

> **Decision-support:** The leakage-safe version is better framed as a decision-support ranking signal to help reviewers prioritise pages, rather than as an automatic refresh decision or a causal predictor of future decline.

### What I will not claim

- I will not claim causal impact.
- I will not claim production-ready generalisation to unseen clients.
- I will not claim that the model "knows" why a page is declining.
- I will not claim that a high score guarantees that a refresh will improve performance.
- I will not present retrospective validation as proof of future business impact.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.